# Exp8.0.3 — Phase-aware hierarchical L1/L2 readout

Aggregation-only notebook. It reads finalizer outputs and does not train models or refit probes. Primary comparison: `l1_fixed250_l2_whole` vs the parameter-matched `l1_capacity_no_phase_l2_whole` control.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_8_0_3_phase_aware_hierarchical_readout' / 'phase_aware_hierarchical_readout_v1'
manifest = json.loads((ART / 'manifest.json').read_text())
method = pd.read_csv(ART / 'method_summary.csv')
method_runs = pd.read_csv(ART / 'method_runs.csv')
probe = pd.read_csv(ART / 'probe_summary.csv')
fusion = pd.read_csv(ART / 'fusion_gain_summary.csv')
overlap = pd.read_csv(ART / 'correctness_overlap_summary.csv')
heads = pd.read_csv(ART / 'trained_head_summary.csv')
paired = pd.read_csv(ART / 'paired_delta_summary.csv')
phase = pd.read_csv(ART / 'phase_structure_summary.csv')
history = pd.read_csv(ART / 'history_runs.csv')
manifest

## Native Linear and same-evidence output-LIF performance

In [ ]:
cols = [
    'method', 'native_test_ba_mean', 'native_test_ba_std',
    'lif_test_ba_mean', 'lif_test_ba_std', 'lif_penalty_mean',
    'accumulator_equivalence_error_mean', 'parameter_count_mean',
    'l1_score_rms_fraction_mean',
]
display(method[cols].sort_values('native_test_ba_mean', ascending=False))

In [ ]:
plot = method.set_index('method')[['native_test_ba_mean', 'lif_test_ba_mean']]
ax = plot.plot(kind='bar', figsize=(10, 4), rot=20)
ax.set_ylabel('Balanced accuracy')
ax.set_title('Exp8.0.3 native Linear vs same-evidence LIF')
plt.tight_layout()

## Paired seed deltas — primary mechanism test

In [ ]:
display(paired)
display(method_runs.pivot(index='seed', columns='method', values='native_test_ba'))

## Frozen representation probes

In [ ]:
focus = probe[probe.feature.isin([
    'l1_fixed250', 'l2_whole', 'l1fixed250_l2whole',
    'l2_fixed250', 'l1_l2_fixed250'
])][['method', 'feature', 'test_ba_mean', 'test_ba_std', 'train_test_gap_mean']]
display(focus.sort_values(['feature', 'test_ba_mean'], ascending=[True, False]))

## Does the large L1 bank actually learn phase-specific weights?

In [ ]:
display(phase[['method', 'mean_pairwise_cosine_mean', 'mean_adjacent_cosine_mean', 'between_bin_weight_rms_mean']])
display(heads)

## Complementarity and post-hoc fusion diagnostics

In [ ]:
display(fusion)
display(overlap[(overlap.split == 'test') & (overlap.aggregation.isin(['fixed250', 'whole']))])

## Training curves — method-level view only

In [ ]:
curve = history.groupby(['method', 'epoch'], as_index=False)[['train_ba', 'val_ba', 'train_loss', 'val_loss']].mean()
fig, ax = plt.subplots(figsize=(10, 4))
for name, frame in curve.groupby('method'):
    ax.plot(frame.epoch, frame.val_ba, label=name)
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean validation BA')
ax.set_title('Exp8.0.3 validation BA')
ax.legend()
plt.tight_layout()